# PanoGS café panorama: Kaggle depth-to-splat comparison

This notebook makes a **separate comparison PLY** from `comfy_cafe_2k.hdr` with the open-source BLUNT + DA360 panorama pipeline. It does not modify the source HDR or call a paid API. The model infers depth from the panorama; it cannot recover the true hidden geometry behind the pillar, chairs, stairs, or other occlusions.

## Kaggle setup
1. In Kaggle, create a **private dataset** containing `comfy_cafe_2k.hdr`, then attach it to this notebook with **Add Input**.
2. In notebook settings, enable **Internet** (to install the open-source code and fetch its model weights) and choose **GPU** if Kaggle offers one.
3. Run the cells top to bottom. The final PLY appears in `/kaggle/working`; save/download it from Kaggle's notebook output files.
4. Put the downloaded PLY in this repo's `output` folder, then run `panogs view output/comfy_cafe_ai.ply`.

Kaggle GPU availability and free usage quotas can vary. The source HDR is uploaded to your Kaggle input dataset, so keep that dataset private if the panorama is not public.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
import numpy as np
import cv2
from PIL import Image
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

SOURCE_HDR = Path('/kaggle/input/datasets/siddhug01/comfy-cafe-2k/comfy_cafe_2k.hdr')
if not SOURCE_HDR.is_file():
    candidates = list(Path('/kaggle/input').rglob('comfy_cafe_2k.hdr'))
else:
    candidates = [SOURCE_HDR]
if not candidates:
    candidates = [p for p in Path('/kaggle/input').rglob('*') if p.suffix.lower() in {'.hdr', '.exr'}]
if len(candidates) != 1:
    raise FileNotFoundError(f'Expected exactly one attached .hdr/.exr panorama; found: {candidates}')
SOURCE_HDR = candidates[0]
print('Source HDR:', SOURCE_HDR)
print('Source size:', SOURCE_HDR.stat().st_size / (1024**2), 'MiB')

## Build a depth-model preview
DA360 accepts a display image. This cell creates a separate Reinhard-mapped JPEG for depth inference; it never overwrites the HDR. The generated Gaussian colors will be display-referred, not the original HDR radiance.

In [ ]:
hdr_bgr = cv2.imread(str(SOURCE_HDR), cv2.IMREAD_UNCHANGED)
if hdr_bgr is None:
    raise RuntimeError(f'OpenCV could not read {SOURCE_HDR}')
hdr_rgb = cv2.cvtColor(hdr_bgr[..., :3], cv2.COLOR_BGR2RGB).astype(np.float32)
hdr_rgb = np.nan_to_num(hdr_rgb, nan=0.0, posinf=0.0, neginf=0.0)
luma = np.maximum(hdr_rgb @ np.array([0.2126, 0.7152, 0.0722], np.float32), 1e-6)
log_average = float(np.exp(np.mean(np.log(luma))))
exposed = hdr_rgb * (0.18 / max(log_average, 1e-6)) * 1.4
reinhard = np.maximum(exposed, 0) / (1.0 + np.maximum(exposed, 0))
srgb = np.where(reinhard <= 0.0031308, reinhard * 12.92, 1.055 * np.power(reinhard, 1 / 2.4) - 0.055)
preview = Path('/kaggle/working/comfy_cafe_depth_input.jpg')
Image.fromarray(np.clip(srgb * 255, 0, 255).astype(np.uint8)).save(preview, quality=96, subsampling=0)
print('Depth input:', preview, 'size:', Image.open(preview).size)
display(Image.open(preview).resize((1024, 512)))

## Install the open-source pipeline
Kaggle commonly includes PyTorch with CUDA support. Keep that build and install the remaining BLUNT dependencies. Internet must be enabled.

In [ ]:
repo = Path('/kaggle/working/blunt')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SonnyC56/blunt.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy', 'Pillow', 'transformers', 'huggingface_hub', 'scipy', 'opencv-python-headless'], check=True)
print('BLUNT checkout:', repo)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Download the official DA360 checkpoint
BLUNT's automatic Hugging Face checkpoint URL currently fails with HTTP 401. The official DA360 project links its pretrained checkpoint from Google Drive. Download it once here and pass it explicitly to BLUNT.

In [ ]:
checkpoint_dir = Path('/kaggle/working/da360_checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_matches = list(checkpoint_dir.rglob('DA360_large.pth'))
if not checkpoint_matches:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'gdown'], check=True)
    drive_folder = 'https://drive.google.com/drive/folders/1FMLWZfJ_IPKOa_cEbVqrq8_BRkl3oB_2'
    subprocess.run([sys.executable, '-m', 'gdown', '--folder', drive_folder, '-O', str(checkpoint_dir)], check=True)
    checkpoint_matches = list(checkpoint_dir.rglob('DA360_large.pth'))
if not checkpoint_matches:
    raise FileNotFoundError(f'DA360_large.pth not found in downloaded official checkpoint folder: {list(checkpoint_dir.rglob("*.pth"))}')
DA360_CHECKPOINT = checkpoint_matches[0]
print('Using official DA360 checkpoint:', DA360_CHECKPOINT, f'({DA360_CHECKPOINT.stat().st_size / (1024**3):.2f} GiB)')

## Generate the café PLY
The 2048 output grid targets roughly two million splats, close to the density of the current PanoGS café result. Increase `RESOLUTION` only after checking viewer performance.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU in notebook Settings > Accelerator, then restart and run again.')
output_ply = Path('/kaggle/working/comfy_cafe_ai.ply')
cmd = [
    sys.executable, str(repo / 'blunt.py'), str(preview),
    '--mode', '360', '--engine', 'da360', '--device', 'cuda',
    '--da360-checkpoint', str(DA360_CHECKPOINT),
    '--resolution', '2048', '-o', str(output_ply),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=repo, check=True)
if not output_ply.is_file() or output_ply.stat().st_size == 0:
    raise RuntimeError('Pipeline completed without a non-empty PLY output')
print(f'Created {output_ply} ({output_ply.stat().st_size / (1024**2):.1f} MiB)')

In [ ]:
manifest = {
    'source_hdr': SOURCE_HDR.name,
    'depth_preview': preview.name,
    'method': 'BLUNT panorama mode with DA360 depth',
    'output_ply': output_ply.name,
    'resolution': 2048,
    'note': 'AI-estimated depth and inferred splat geometry; not a metrically exact reconstruction. HDR source radiance is not encoded in this PLY.'
}
Path('/kaggle/working/comfy_cafe_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print('Download comfy_cafe_ai.ply and comfy_cafe_manifest.json from the Kaggle notebook output files.')